# Train MicroRTS agents on Colab

<a href="https://colab.research.google.com/github/Crunchy000/petting-zoo-micro-rts/blob/main/training/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook trains a gridnet agent for [MicroRTS for PettingZoo](https://github.com/Crunchy000/petting-zoo-micro-rts) with PPO and action masking, then records games as HTML replays.

It uses `training/train_jax.py`, which runs the games, the scripted opponents and the network together on the GPU. See [`training/README.md`](https://github.com/Crunchy000/petting-zoo-micro-rts/blob/main/training/README.md) for what every option does.

1. **Runtime → Change runtime type → L4 or T4 GPU** (any GPU works; a CPU runtime works too, but far slower).
2. **Runtime → Run all.**
3. When it finishes, the last cell downloads `microrts_agent.zip` (the trained model and replays). Upload that zip to the project thread and the games go on the website.

Training runs in two stages. First it learns on an 8×8 map, starting by imitating the scripted bots and then improving against them and against older copies of itself. Then it carries on the 16×16 map used on the website. The time limits below fit a free Colab session; raise them if you have Colab Pro.

In [ ]:
#@title Settings
STAGE1_MINUTES = 40  #@param {type:"integer"}
STAGE2_MINUTES = 80  #@param {type:"integer"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

Save checkpoints to Google Drive so a disconnected session can resume (`--resume`) instead of starting over.

In [ ]:
import os
OUT = "/content/runs"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/microrts_runs"
os.makedirs(OUT, exist_ok=True)
print("checkpoints go to", OUT)

In [ ]:
!git clone -q https://github.com/Crunchy000/petting-zoo-micro-rts /content/petting-zoo-micro-rts || git -C /content/petting-zoo-micro-rts pull -q
%cd /content/petting-zoo-micro-rts
!pip install -q -e ".[jax]"   # Colab GPU runtimes already have JAX with CUDA
!python -c "import jax; print(jax.devices())"
!nproc; nvidia-smi -L || true

## Stage 1: 8×8 map, warm start by imitating the bots

In [ ]:
!python training/train_jax.py --map maps/8x8/basesWorkers8x8A.xml --out {OUT}/8x8 \
    --total-steps 30000000 --bc-steps 1000000 --max-steps 1000 --time-limit {STAGE1_MINUTES} --resume

## Stage 2: 16×16 map, starting from the 8×8 agent

In [ ]:
!python training/train_jax.py --map maps/16x16/basesWorkers16x16.xml --out {OUT}/16x16 \
    --init {OUT}/8x8/latest.pkl --total-steps 200000000 --time-limit {STAGE2_MINUTES} --resume

## Evaluate and record games

In [ ]:
!python training/play.py {OUT}/16x16/latest.pkl --opponent mixed --games 10 --record {OUT}/replays --record-games 3
!python training/play.py {OUT}/16x16/latest.pkl --opponent light --games 10 --record {OUT}/replays --record-games 2
!python training/play.py {OUT}/16x16/latest.pkl --map maps/letMeOut.xml --opponent mixed --games 4 --record {OUT}/replays --record-games 1

In [ ]:
# Watch the first recorded game here
import glob
from IPython.display import HTML, display
replays = sorted(glob.glob(f"{OUT}/replays/*.html"))
print("\n".join(replays))
if replays:
    display(HTML(open(replays[0]).read()))

## Download the results

Upload `microrts_agent.zip` to the project thread.

In [ ]:
import shutil
pack = "/content/microrts_agent"
shutil.rmtree(pack, ignore_errors=True)
os.makedirs(pack)
for name in ("8x8", "16x16"):
    shutil.copy(f"{OUT}/{name}/latest.pt", f"{pack}/agent_{name}.pt")   # PyTorch format, for the website and play.py
    shutil.copy(f"{OUT}/{name}/latest.pkl", f"{pack}/agent_{name}.pkl") # JAX format, with optimiser and snapshots
for name in ("8x8", "16x16"):
    shutil.copy(f"{OUT}/{name}/metrics.csv", f"{pack}/metrics_{name}.csv")
shutil.copytree(f"{OUT}/replays", f"{pack}/replays")
shutil.make_archive("/content/microrts_agent", "zip", pack)
from google.colab import files
files.download("/content/microrts_agent.zip")